# 01. Data Exploration — MLB Historical Odds 2010-2021

Explore the dataset: distributions, quality checks, key statistics.
Foundation for all subsequent analysis.

In [ ]:
# Google Colab: uncomment to install dependencies
# !pip install openpyxl scipy

import os
import sys
import warnings

warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('Ready!')

## 1. Load Raw Data

Load all 11 seasons, inspect raw structure before filtering.

In [ ]:
from src.data_loader import (
    add_derived_odds,
    apply_betting_filters,
    apply_data_filters,
    load_all_seasons,
)

# Load raw (no filters yet)
games_raw = load_all_seasons()
print(f'Raw games loaded: {len(games_raw)}')
print(f'Columns ({len(games_raw.columns)}): {list(games_raw.columns)}')
print(f'\nSeasons: {sorted(games_raw["season"].unique())}')
print(f'Date range: {games_raw["date"].min()} to {games_raw["date"].max()}')
games_raw.head(3)

In [ ]:
# Games per season
season_counts = games_raw.groupby('season').size().reset_index(name='games')
print('Games per season:')
for _, row in season_counts.iterrows():
    print(f"  {int(row['season'])}: {int(row['games'])} games")
print(f"  Total: {season_counts['games'].sum()} games")
print(f"  Avg per season: {season_counts['games'].mean():.0f}")

## 2. Data Quality Checks

In [ ]:
# Missing values check
print('Missing values in key columns:')
key_cols = ['date', 'home_team', 'away_team', 'home_pitcher', 'away_pitcher',
            'home_final', 'away_final', 'home_close_ml', 'away_close_ml',
            'close_ou', 'home_run_line']
for col in key_cols:
    missing = games_raw[col].isna().sum()
    zeros = (games_raw[col] == 0).sum() if games_raw[col].dtype in ['int64', 'float64'] else 0
    print(f'  {col:>20s}: {missing:5d} NaN, {zeros:5d} zeros')

# Inning scores: check for unexpected NaN
print('\nInning score NaN counts (should be 0):')
for side in ['away', 'home']:
    for i in range(1, 10):
        col = f'{side}_inn_{i}'
        n = games_raw[col].isna().sum()
        if n > 0:
            print(f'  {col}: {n} NaN')
print('  (none found = good)' if all(games_raw[f'{s}_inn_{i}'].isna().sum() == 0 for s in ['away', 'home'] for i in range(1, 10)) else '')

In [ ]:
# Verify inning sum vs final score
# If they don't match, game went to extra innings
games_raw['away_inn_sum'] = sum(games_raw[f'away_inn_{i}'] for i in range(1, 10))
games_raw['home_inn_sum'] = sum(games_raw[f'home_inn_{i}'] for i in range(1, 10))

away_mismatch = (games_raw['away_inn_sum'] != games_raw['away_final']).sum()
home_mismatch = (games_raw['home_inn_sum'] != games_raw['home_final']).sum()

print(f'Away: inning sum != final score: {away_mismatch} games ({away_mismatch/len(games_raw):.1%})')
print(f'Home: inning sum != final score: {home_mismatch} games ({home_mismatch/len(games_raw):.1%})')
print('=> These are extra-inning games (runs scored after 9th inning)')

# Check for ties (should not exist in MLB)
ties = (games_raw['home_final'] == games_raw['away_final']).sum()
print(f'\nTies (home_final == away_final): {ties} (should be 0 or very few suspended games)')

## 3. Apply Filters and Inspect Impact

In [ ]:
# Apply data quality filters
games = apply_data_filters(games_raw.copy())
games = add_derived_odds(games)

print(f'After data filters: {len(games)} games (removed {len(games_raw) - len(games)})')
print('\nSoft flags:')
print(f'  Colorado games:    {games["involves_col"].sum():5d} ({games["involves_col"].mean():.1%})')
print(f'  September games:   {games["is_september"].sum():5d} ({games["is_september"].mean():.1%})')
print(f'  Extreme lines:     {games["is_extreme_line"].sum():5d} ({games["is_extreme_line"].mean():.1%})')
print(f'  Extra innings:     {games["is_extra_innings"].sum():5d} ({games["is_extra_innings"].mean():.1%})')
print(f'  Early season:      {games["is_early_season"].sum():5d} ({games["is_early_season"].mean():.1%})')

# Apply betting filters
bettable = apply_betting_filters(games)
print(f'\nBettable games: {len(bettable)} ({len(bettable)/len(games):.1%} of filtered)')

## 4. Scoring Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total runs distribution
ax = axes[0]
total_runs = bettable['total_runs']
ax.hist(total_runs, bins=range(0, 35), color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(total_runs.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {total_runs.mean():.1f}')
ax.axvline(total_runs.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {total_runs.median():.0f}')
ax.set_xlabel('Total Runs')
ax.set_ylabel('Count')
ax.set_title('Total Runs per Game')
ax.legend()

# Home vs Away final scores
ax = axes[1]
ax.hist(bettable['home_final'], bins=range(0, 20), alpha=0.6, color='#388e3c', label='Home', edgecolor='white')
ax.hist(bettable['away_final'], bins=range(0, 20), alpha=0.6, color='#d32f2f', label='Away', edgecolor='white')
ax.set_xlabel('Runs Scored')
ax.set_ylabel('Count')
ax.set_title('Runs Scored: Home vs Away')
ax.legend()

# Run margin (home perspective)
ax = axes[2]
margin = bettable['home_final'] - bettable['away_final']
ax.hist(margin, bins=range(-15, 16), color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(0, color='red', linestyle='-', linewidth=2)
ax.set_xlabel('Home Margin (positive = home win)')
ax.set_ylabel('Count')
ax.set_title(f'Run Margin Distribution (Home Win%: {bettable["home_win"].mean():.1%})')

plt.tight_layout()
plt.show()

print('Scoring stats (bettable games):')
print(f'  Avg total runs/game: {total_runs.mean():.2f}')
print(f'  Avg home runs: {bettable["home_final"].mean():.2f}')
print(f'  Avg away runs: {bettable["away_final"].mean():.2f}')
print(f'  Home win rate: {bettable["home_win"].mean():.1%}')

## 5. Odds Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Moneyline distribution
ax = axes[0]
ax.hist(bettable['home_close_ml'], bins=80, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(0, color='red', linestyle='-', linewidth=1)
ax.set_xlabel('Home Closing Moneyline')
ax.set_ylabel('Count')
ax.set_title('Moneyline Distribution (Home)')

# Over/Under lines
ax = axes[1]
ax.hist(bettable['close_ou'], bins=40, color='#ff7043', alpha=0.8, edgecolor='white')
ax.axvline(bettable['close_ou'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Mean: {bettable["close_ou"].mean():.1f}')
ax.set_xlabel('Closing O/U Line')
ax.set_ylabel('Count')
ax.set_title('Over/Under Line Distribution')
ax.legend()

# Implied probability vs actual win rate
ax = axes[2]
# Bin implied probabilities and compare to actual outcomes
bettable_temp = bettable.copy()
bettable_temp['impl_bin'] = pd.cut(bettable_temp['home_implied_prob'], bins=20)
cal = bettable_temp.groupby('impl_bin', observed=True).agg(
    implied=('home_implied_prob', 'mean'),
    actual=('home_win', 'mean'),
    count=('home_win', 'count')
).dropna()
ax.scatter(cal['implied'], cal['actual'], s=cal['count']/5, alpha=0.7, color='steelblue')
ax.plot([0.2, 0.8], [0.2, 0.8], 'r--', linewidth=2, label='Perfect calibration')
ax.set_xlabel('Implied Probability (from closing ML)')
ax.set_ylabel('Actual Win Rate')
ax.set_title('Odds Calibration: Implied vs Actual')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Vig analysis
bettable_temp = bettable.copy()
bettable_temp['total_implied'] = bettable_temp['home_implied_prob'] + bettable_temp['away_implied_prob']
bettable_temp['vig'] = bettable_temp['total_implied'] - 1.0

print('Vig (overround) analysis:')
print(f'  Mean vig: {bettable_temp["vig"].mean():.3f} ({bettable_temp["vig"].mean()*100:.1f}%)')
print(f'  Median vig: {bettable_temp["vig"].median():.3f}')
print(f'  Min vig: {bettable_temp["vig"].min():.3f}')
print(f'  Max vig: {bettable_temp["vig"].max():.3f}')

# Vig by favorite strength
print('\nVig by favorite strength (home ML):')
bins = [(-400, -250, 'Heavy fav'), (-250, -150, 'Mod fav'), (-150, -100, 'Slight fav'),
        (100, 150, 'Slight dog'), (150, 250, 'Mod dog'), (250, 400, 'Heavy dog')]
for lo, hi, label in bins:
    mask = (bettable_temp['home_close_ml'] >= lo) & (bettable_temp['home_close_ml'] < hi)
    if mask.sum() > 0:
        avg_vig = bettable_temp.loc[mask, 'vig'].mean()
        print(f'  {label:>12s} ({lo:+d} to {hi:+d}): vig={avg_vig:.3f}, n={mask.sum()}')

## 6. First Inning Scoring (YRFI Preview)

In [ ]:
# First inning runs
bettable_temp = bettable.copy()
bettable_temp['first_inning_runs'] = bettable_temp['away_inn_1'] + bettable_temp['home_inn_1']
bettable_temp['yrfi'] = (bettable_temp['first_inning_runs'] > 0).astype(int)

print('First Inning Scoring (YRFI Analysis):')
print(f'  YRFI rate (overall): {bettable_temp["yrfi"].mean():.1%}')
print(f'  Avg runs in 1st inning: {bettable_temp["first_inning_runs"].mean():.2f}')
print()

# YRFI by season
print('YRFI rate by season:')
yrfi_by_season = bettable_temp.groupby('season')['yrfi'].mean()
for season, rate in yrfi_by_season.items():
    print(f'  {int(season)}: {rate:.1%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# YRFI rate by season
ax = axes[0]
ax.bar(yrfi_by_season.index.astype(int), yrfi_by_season.values * 100, color='steelblue', alpha=0.8)
ax.axhline(bettable_temp['yrfi'].mean() * 100, color='red', linestyle='--', linewidth=2,
           label=f'Overall: {bettable_temp["yrfi"].mean():.1%}')
ax.set_xlabel('Season')
ax.set_ylabel('YRFI Rate (%)')
ax.set_title('YRFI Rate by Season')
ax.legend()

# First inning runs distribution
ax = axes[1]
ax.hist(bettable_temp['first_inning_runs'], bins=range(0, 12), color='#ff7043', alpha=0.8, edgecolor='white')
ax.set_xlabel('Total Runs in 1st Inning')
ax.set_ylabel('Count')
ax.set_title('1st Inning Runs Distribution')

plt.tight_layout()
plt.show()

## 7. First 5 Innings Analysis (F5 Preview)

In [ ]:
# F5 scoring
bettable_temp = bettable.copy()
away_f5 = sum(bettable_temp[f'away_inn_{i}'] for i in range(1, 6))
home_f5 = sum(bettable_temp[f'home_inn_{i}'] for i in range(1, 6))
bettable_temp['f5_total'] = away_f5 + home_f5
bettable_temp['f5_home_lead'] = home_f5 - away_f5

# F5 results
f5_home_win = (home_f5 > away_f5).mean()
f5_push = (home_f5 == away_f5).mean()
f5_away_win = (away_f5 > home_f5).mean()

print('First 5 Innings Analysis:')
print(f'  Avg F5 total runs: {bettable_temp["f5_total"].mean():.2f}')
print(f'  F5 Home win: {f5_home_win:.1%}')
print(f'  F5 Push (tie): {f5_push:.1%}')
print(f'  F5 Away win: {f5_away_win:.1%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F5 total distribution
ax = axes[0]
ax.hist(bettable_temp['f5_total'], bins=range(0, 20), color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(bettable_temp['f5_total'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Mean: {bettable_temp["f5_total"].mean():.1f}')
ax.set_xlabel('F5 Total Runs')
ax.set_ylabel('Count')
ax.set_title('First 5 Innings Total Runs')
ax.legend()

# F5 total by season
ax = axes[1]
f5_by_season = bettable_temp.groupby('season')['f5_total'].mean()
ax.bar(f5_by_season.index.astype(int), f5_by_season.values, color='#ff7043', alpha=0.8)
ax.axhline(bettable_temp['f5_total'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Overall: {bettable_temp["f5_total"].mean():.1f}')
ax.set_xlabel('Season')
ax.set_ylabel('Avg F5 Total Runs')
ax.set_title('F5 Total by Season')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Team Analysis

In [ ]:
# Home win rate by team (all seasons combined)
home_wr = games.groupby('home_team').agg(
    home_games=('home_win', 'count'),
    home_wins=('home_win', 'sum'),
    home_wr=('home_win', 'mean'),
    avg_home_runs=('home_final', 'mean'),
    avg_away_runs=('away_final', 'mean'),
).reset_index().sort_values('home_wr', ascending=False)

fig, ax = plt.subplots(figsize=(16, 8))
colors = ['#d32f2f' if t == 'COL' else 'steelblue' for t in home_wr['home_team']]
bars = ax.barh(home_wr['home_team'], home_wr['home_wr'] * 100, color=colors, alpha=0.8)
ax.axvline(50, color='gray', linestyle='--', linewidth=1.5)
ax.set_xlabel('Home Win Rate (%)')
ax.set_title('Home Win Rate by Team (2010-2021, Colorado in red)')
ax.set_xlim(40, 65)
plt.tight_layout()
plt.show()

print(f'Top 5 home teams: {home_wr.head(5)[["home_team", "home_wr"]].to_string(index=False)}')
print(f'\nColorado home WR: {home_wr[home_wr["home_team"]=="COL"]["home_wr"].values[0]:.1%}')

## 9. Inning-by-Inning Scoring Patterns

In [ ]:
# Average runs per inning
away_inn_avgs = [bettable[f'away_inn_{i}'].mean() for i in range(1, 10)]
home_inn_avgs = [bettable[f'home_inn_{i}'].mean() for i in range(1, 10)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Average runs per inning by side
ax = axes[0]
x = range(1, 10)
ax.plot(x, away_inn_avgs, 'o-', color='#d32f2f', linewidth=2, markersize=8, label='Away')
ax.plot(x, home_inn_avgs, 's-', color='#388e3c', linewidth=2, markersize=8, label='Home')
ax.set_xlabel('Inning')
ax.set_ylabel('Average Runs')
ax.set_title('Average Runs per Inning')
ax.legend()
ax.set_xticks(range(1, 10))

# Probability of scoring at least 1 run per inning
ax = axes[1]
away_score_prob = [(bettable[f'away_inn_{i}'] > 0).mean() for i in range(1, 10)]
home_score_prob = [(bettable[f'home_inn_{i}'] > 0).mean() for i in range(1, 10)]
ax.plot(x, away_score_prob, 'o-', color='#d32f2f', linewidth=2, markersize=8, label='Away')
ax.plot(x, home_score_prob, 's-', color='#388e3c', linewidth=2, markersize=8, label='Home')
ax.set_xlabel('Inning')
ax.set_ylabel('P(score >= 1 run)')
ax.set_title('Probability of Scoring in Each Inning')
ax.legend()
ax.set_xticks(range(1, 10))

plt.tight_layout()
plt.show()

print('Average runs per inning:')
for i in range(1, 10):
    total_avg = away_inn_avgs[i-1] + home_inn_avgs[i-1]
    print(f'  Inning {i}: Away={away_inn_avgs[i-1]:.3f}, Home={home_inn_avgs[i-1]:.3f}, Total={total_avg:.3f}')

## 10. Colorado Effect (Why We Exclude from Betting)

In [ ]:
# Compare Colorado games vs non-Colorado games
col_games = games[games['involves_col']]
non_col = games[~games['involves_col']]

print('Colorado Effect:')
print(f'  Colorado games: {len(col_games)} ({len(col_games)/len(games):.1%})')
print(f'  Non-Colorado games: {len(non_col)}')
print()
print(f'  Total runs/game (COL): {col_games["total_runs"].mean():.2f}')
print(f'  Total runs/game (non-COL): {non_col["total_runs"].mean():.2f}')
print(f'  Difference: {col_games["total_runs"].mean() - non_col["total_runs"].mean():+.2f} runs')
print()

# O/U line accuracy
col_ou_diff = col_games['total_runs'].mean() - col_games['close_ou'].mean()
non_col_ou_diff = non_col['total_runs'].mean() - non_col['close_ou'].mean()
print(f'  Avg O/U line (COL): {col_games["close_ou"].mean():.1f}')
print(f'  Avg O/U line (non-COL): {non_col["close_ou"].mean():.1f}')
print(f'  Actual vs Line (COL): {col_ou_diff:+.2f}')
print(f'  Actual vs Line (non-COL): {non_col_ou_diff:+.2f}')

# YRFI comparison
col_yrfi = (col_games['away_inn_1'] + col_games['home_inn_1'] > 0).mean()
non_col_yrfi = (non_col['away_inn_1'] + non_col['home_inn_1'] > 0).mean()
print(f'\n  YRFI rate (COL): {col_yrfi:.1%}')
print(f'  YRFI rate (non-COL): {non_col_yrfi:.1%}')

## 11. O/U Line vs Actual Total — Can We Beat the Line?

In [ ]:
# Over/Under accuracy check
bt = bettable.copy()
bt['over'] = (bt['total_runs'] > bt['close_ou']).astype(int)
bt['under'] = (bt['total_runs'] < bt['close_ou']).astype(int)
bt['push_ou'] = (bt['total_runs'] == bt['close_ou']).astype(int)

over_rate = bt['over'].mean()
under_rate = bt['under'].mean()
push_rate = bt['push_ou'].mean()

print('Over/Under Results (bettable):')
print(f'  Over:  {over_rate:.1%}')
print(f'  Under: {under_rate:.1%}')
print(f'  Push:  {push_rate:.1%}')

# By O/U line range
print('\nOver rate by O/U line range:')
bt['ou_bin'] = pd.cut(bt['close_ou'], bins=[5, 7, 8, 8.5, 9, 9.5, 10, 11, 15])
ou_stats = bt.groupby('ou_bin', observed=True).agg(
    games=('over', 'count'),
    over_rate=('over', 'mean'),
    avg_total=('total_runs', 'mean'),
).reset_index()
for _, row in ou_stats.iterrows():
    print(f'  Line {row["ou_bin"]}: Over={row["over_rate"]:.1%}, '
          f'Avg total={row["avg_total"]:.1f}, n={int(row["games"])}')

# By season
print('\nOver rate by season:')
for season in sorted(bt['season'].unique()):
    s = bt[bt['season'] == season]
    or_s = s['over'].mean()
    avg_ou = s['close_ou'].mean()
    avg_total = s['total_runs'].mean()
    print(f'  {int(season)}: Over={or_s:.1%}, Avg line={avg_ou:.1f}, Avg actual={avg_total:.1f}')

## 12. Summary Statistics

In [ ]:
print('=' * 70)
print('DATA EXPLORATION SUMMARY')
print('=' * 70)
print(f'''
Dataset:
  Seasons: 11 (2010-2019, 2021, excl. 2020 COVID)
  Raw games: {len(games_raw)}
  After data filters: {len(games)}
  Bettable games: {len(bettable)}

Scoring:
  Avg total runs/game: {bettable['total_runs'].mean():.2f}
  Avg home runs: {bettable['home_final'].mean():.2f}
  Avg away runs: {bettable['away_final'].mean():.2f}
  Home win rate: {bettable['home_win'].mean():.1%}

Markets (bettable games):
  YRFI rate: {(bettable['away_inn_1'] + bettable['home_inn_1'] > 0).mean():.1%}
  Avg F5 total: {(sum(bettable[f'away_inn_{i}'] for i in range(1, 6)) + sum(bettable[f'home_inn_{i}'] for i in range(1, 6))).mean():.2f}
  Over rate: {bt['over'].mean():.1%}

Odds:
  Avg home implied prob: {bettable['home_implied_prob'].mean():.3f}
  Avg vig: {(bettable['home_implied_prob'] + bettable['away_implied_prob']).mean() - 1:.3f}

Filters removed:
  Colorado games: {games['involves_col'].sum()} (flagged, not bet on)
  September games: {games['is_september'].sum()} (flagged, not bet on)
  Extreme lines: {games['is_extreme_line'].sum()} (flagged, not bet on)
  Extra innings: {games['is_extra_innings'].sum()} (flagged for awareness)
''')